In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [2]:
import tensorflow as tf
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.applications.resnet50 import preprocess_input
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D
from tensorflow.keras.optimizers import Adam

2026-11-22 05:12:41.918730: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1795324362.115361      57 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1795324362.159069      57 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1795324362.561400      57 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1795324362.561462      57 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1795324362.561466      57 computation_placer.cc:177] computation placer alr

In [3]:
train_gen = ImageDataGenerator(
    preprocessing_function=preprocess_input,
    rotation_range=20,
    width_shift_range=0.1,
    height_shift_range=0.1,
    zoom_range=0.2,
    horizontal_flip=True
)

val_gen = ImageDataGenerator(
    preprocessing_function=preprocess_input
)

In [4]:
train_data = train_gen.flow_from_directory(
    "/kaggle/input/e-waste-image-dataset/modified-dataset/train",
    target_size=(224,224),
    batch_size=32,
    class_mode="categorical"
)

Found 2400 images belonging to 10 classes.


In [5]:
val_data = val_gen.flow_from_directory(
    "/kaggle/input/e-waste-image-dataset/modified-dataset/val",
    target_size=(224,224),
    batch_size=32,
    class_mode="categorical"
)

Found 300 images belonging to 10 classes.


In [6]:
base = ResNet50(
    weights="imagenet",
    include_top=False,
    input_shape=(224,224,3)
)

base.trainable = False

I0000 00:00:1795324377.089325      57 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13756 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1795324377.095341      57 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13756 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5


94765736/94765736 ━━━━━━━━━━━━━━━━━━━━ 6s 0us/step


In [7]:
x = GlobalAveragePooling2D()(base.output)
x = Dense(128, activation="relu")(x)
out = Dense(train_data.num_classes, activation="softmax")(x)

model = Model(inputs=base.input, outputs=out)

In [8]:
model.compile(
    optimizer=Adam(1e-4),
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

In [9]:
history = model.fit(
    train_data,
    validation_data=val_data,
    epochs=10
)

/usr/local/lib/python3.12/dist-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


Epoch 1/10


I0000 00:00:1795324398.392463     142 service.cc:152] XLA service 0x7881e4002260 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1795324398.392519     142 service.cc:160]   StreamExecutor device (0): Tesla T4, Compute Capability 7.5
I0000 00:00:1795324398.392527     142 service.cc:160]   StreamExecutor device (1): Tesla T4, Compute Capability 7.5
I0000 00:00:1795324400.353338     142 cuda_dnn.cc:529] Loaded cuDNN version 91002


 2/75 ━━━━━━━━━━━━━━━━━━━━ 6s 84ms/step - accuracy: 0.0781 - loss: 3.0163  

I0000 00:00:1795324405.303731     142 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


75/75 ━━━━━━━━━━━━━━━━━━━━ 188s 2s/step - accuracy: 0.3200 - loss: 2.0669 - val_accuracy: 0.8600 - val_loss: 0.6558
Epoch 2/10
75/75 ━━━━━━━━━━━━━━━━━━━━ 28s 366ms/step - accuracy: 0.8485 - loss: 0.6234 - val_accuracy: 0.9400 - val_loss: 0.3118
Epoch 3/10
75/75 ━━━━━━━━━━━━━━━━━━━━ 28s 367ms/step - accuracy: 0.9103 - loss: 0.3611 - val_accuracy: 0.9367 - val_loss: 0.2364
Epoch 4/10
75/75 ━━━━━━━━━━━━━━━━━━━━ 28s 372ms/step - accuracy: 0.9304 - loss: 0.2741 - val_accuracy: 0.9567 - val_loss: 0.1877
Epoch 5/10
75/75 ━━━━━━━━━━━━━━━━━━━━ 28s 369ms/step - accuracy: 0.9307 - loss: 0.2409 - val_accuracy: 0.9500 - val_loss: 0.1803
Epoch 6/10
75/75 ━━━━━━━━━━━━━━━━━━━━ 27s 358ms/step - accuracy: 0.9461 - loss: 0.1935 - val_accuracy: 0.9667 - val_loss: 0.1486
Epoch 7/10
75/75 ━━━━━━━━━━━━━━━━━━━━ 27s 362ms/step - accuracy: 0.9568 - loss: 0.1550 - val_accuracy: 0.9667 - val_loss: 0.1389
Epoch 8/10
75/75 ━━━━━━━━━━━━━━━━━━━━ 27s 359ms/step - accuracy: 0.9604 - loss: 0.1382 - val_accuracy: 0.9667 

In [10]:
test_gen = ImageDataGenerator(
    preprocessing_function=preprocess_input
)

test_data = test_gen.flow_from_directory(
    "/kaggle/input/e-waste-image-dataset/modified-dataset/test",
    target_size=(224,224),
    batch_size=32,
    class_mode="categorical",
    shuffle=False
)


Found 300 images belonging to 10 classes.


In [11]:
test_loss, test_acc = model.evaluate(test_data)
print("Test accuracy:", test_acc)

10/10 ━━━━━━━━━━━━━━━━━━━━ 14s 2s/step - accuracy: 0.9310 - loss: 0.2027
Test accuracy: 0.9466666579246521


In [12]:
import numpy as np
from sklearn.metrics import classification_report, confusion_matrix

y_true = test_data.classes
y_pred = np.argmax(model.predict(test_data), axis=1)

print(confusion_matrix(y_true, y_pred))
print(classification_report(y_true, y_pred, target_names=test_data.class_indices.keys()))

10/10 ━━━━━━━━━━━━━━━━━━━━ 8s 466ms/step
[[27  0  0  0  0  1  1  0  0  1]
 [ 2 27  0  0  0  1  0  0  0  0]
 [ 0  0 28  0  0  0  0  1  1  0]
 [ 0  0  0 29  0  0  0  0  1  0]
 [ 1  0  0  0 29  0  0  0  0  0]
 [ 1  0  0  0  0 28  1  0  0  0]
 [ 1  0  0  0  0  0 29  0  0  0]
 [ 1  0  0  0  0  0  0 29  0  0]
 [ 0  0  2  0  0  0  0  0 28  0]
 [ 0  0  0  0  0  0  0  0  0 30]]
                 precision    recall  f1-score   support

        Battery       0.82      0.90      0.86        30
       Keyboard       1.00      0.90      0.95        30
      Microwave       0.93      0.93      0.93        30
         Mobile       1.00      0.97      0.98        30
          Mouse       1.00      0.97      0.98        30
            PCB       0.93      0.93      0.93        30
         Player       0.94      0.97      0.95        30
        Printer       0.97      0.97      0.97        30
     Television       0.93      0.93      0.93        30
Washing Machine       0.97      1.00      0.98        30


In [13]:
#model.save("ewaste_model.keras")

In [14]:
for layer in base.layers:
    if "conv5" in layer.name:
        layer.trainable = True
    else:
        layer.trainable = False

In [15]:
model.compile(
    optimizer=Adam(learning_rate=1e-5),
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

In [16]:
history_fine = model.fit(
    train_data,
    validation_data=val_data,
    epochs=5
)

Epoch 1/5
75/75 ━━━━━━━━━━━━━━━━━━━━ 51s 439ms/step - accuracy: 0.8942 - loss: 0.3125 - val_accuracy: 0.9533 - val_loss: 0.1699
Epoch 2/5
75/75 ━━━━━━━━━━━━━━━━━━━━ 28s 365ms/step - accuracy: 0.9682 - loss: 0.1351 - val_accuracy: 0.9567 - val_loss: 0.1459
Epoch 3/5
75/75 ━━━━━━━━━━━━━━━━━━━━ 28s 365ms/step - accuracy: 0.9708 - loss: 0.1092 - val_accuracy: 0.9700 - val_loss: 0.1301
Epoch 4/5
75/75 ━━━━━━━━━━━━━━━━━━━━ 28s 366ms/step - accuracy: 0.9793 - loss: 0.0896 - val_accuracy: 0.9700 - val_loss: 0.1285
Epoch 5/5
75/75 ━━━━━━━━━━━━━━━━━━━━ 28s 373ms/step - accuracy: 0.9860 - loss: 0.0670 - val_accuracy: 0.9733 - val_loss: 0.1223


In [17]:
test_loss, test_acc = model.evaluate(test_data)
print("Test accuracy:", test_acc)

10/10 ━━━━━━━━━━━━━━━━━━━━ 1s 85ms/step - accuracy: 0.9495 - loss: 0.1398
Test accuracy: 0.95333331823349


In [18]:
import numpy as np
from sklearn.metrics import classification_report, confusion_matrix

y_true = test_data.classes
y_pred = np.argmax(model.predict(test_data), axis=1)

print(confusion_matrix(y_true, y_pred))
print(classification_report(y_true, y_pred, target_names=test_data.class_indices.keys()))

10/10 ━━━━━━━━━━━━━━━━━━━━ 8s 465ms/step
[[27  0  0  0  0  1  1  0  0  1]
 [ 1 29  0  0  0  0  0  0  0  0]
 [ 0  0 30  0  0  0  0  0  0  0]
 [ 0  0  0 28  0  0  0  0  2  0]
 [ 0  0  0  0 30  0  0  0  0  0]
 [ 1  0  0  0  0 28  1  0  0  0]
 [ 1  0  0  0  0  0 29  0  0  0]
 [ 0  0  0  0  0  0  1 29  0  0]
 [ 0  0  2  0  0  0  1  0 27  0]
 [ 1  0  0  0  0  0  0  0  0 29]]
                 precision    recall  f1-score   support

        Battery       0.87      0.90      0.89        30
       Keyboard       1.00      0.97      0.98        30
      Microwave       0.94      1.00      0.97        30
         Mobile       1.00      0.93      0.97        30
          Mouse       1.00      1.00      1.00        30
            PCB       0.97      0.93      0.95        30
         Player       0.88      0.97      0.92        30
        Printer       1.00      0.97      0.98        30
     Television       0.93      0.90      0.92        30
Washing Machine       0.97      0.97      0.97        30


In [19]:
for layer in base.layers[-90:]:
    print(layer.name, layer.trainable)

conv4_block1_2_bn False
conv4_block1_2_relu False
conv4_block1_0_conv False
conv4_block1_3_conv False
conv4_block1_0_bn False
conv4_block1_3_bn False
conv4_block1_add False
conv4_block1_out False
conv4_block2_1_conv False
conv4_block2_1_bn False
conv4_block2_1_relu False
conv4_block2_2_conv False
conv4_block2_2_bn False
conv4_block2_2_relu False
conv4_block2_3_conv False
conv4_block2_3_bn False
conv4_block2_add False
conv4_block2_out False
conv4_block3_1_conv False
conv4_block3_1_bn False
conv4_block3_1_relu False
conv4_block3_2_conv False
conv4_block3_2_bn False
conv4_block3_2_relu False
conv4_block3_3_conv False
conv4_block3_3_bn False
conv4_block3_add False
conv4_block3_out False
conv4_block4_1_conv False
conv4_block4_1_bn False
conv4_block4_1_relu False
conv4_block4_2_conv False
conv4_block4_2_bn False
conv4_block4_2_relu False
conv4_block4_3_conv False
conv4_block4_3_bn False
conv4_block4_add False
conv4_block4_out False
conv4_block5_1_conv False
conv4_block5_1_bn False
conv4_bloc

In [20]:
import json

model.save("ewaste_model.keras")
with open("class_indices.json", "w") as f:
    json.dump(train_data.class_indices, f)

print("Saved ewaste_model.keras and class_indices.json!")

Saved ewaste_model.keras and class_indices.json!
